In [5]:
# =============================================================================
# FUNCTION 1 - BLACK-BOX OPTIMISATION
# =============================================================================
# Prepares the harmonised modelling dataset for the whole BBO pipeline
# and writes it to data/data.csv, combining initial observations from
# the original .npy files (Functions 1-8) with all subsequent
# optimisation rounds parsed from data/new_data/inputs.txt and
# outputs.txt. All functions are MAXIMISATION problems: higher y wins.

In [6]:
# =============================================================================
# IMPORTS: LOAD IMMUTABLE INITIAL BBO DATA FROM .NPY FILES
# =============================================================================
# Imports pathlib, NumPy and pandas, used to read the raw .npy arrays
# and build the initial observations dataframe.

from pathlib import Path
import numpy as np
import pandas as pd

In [7]:
# =============================================================================
# LOAD FUNCTION 1 FROM HARMONISED DATASET
# =============================================================================
# Loads the original per-function input/output arrays from
# ./initial_data/function_N/initial_inputs.npy and initial_outputs.npy
# for Functions 1-8. Validates each array's shape, dimensionality and
# float64 dtype before combining them into a single dataframe with
# columns round, function, x1..x8, y. Values remain float64 throughout.

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

INITIAL_DATA_DIR = Path("../data/initial_data")

FUNCTION_DIMS = {
    1: 2,
    2: 2,
    3: 3,
    4: 4,
    5: 4,
    6: 5,
    7: 6,
    8: 8,
}


# ------------------------------------------------------------
# 2. LOAD EACH FUNCTION
# ------------------------------------------------------------

initial_frames = []

for function, expected_dim in FUNCTION_DIMS.items():

    function_dir = INITIAL_DATA_DIR / f"function_{function}"

    inputs_path = function_dir / "initial_inputs.npy"
    outputs_path = function_dir / "initial_outputs.npy"

    # Load the original NumPy arrays exactly as stored.
    X = np.load(
        inputs_path,
        allow_pickle=False
    )

    y = np.load(
        outputs_path,
        allow_pickle=False
    )


    # --------------------------------------------------------
    # 3. VALIDATE SOURCE ARRAYS
    # --------------------------------------------------------

    if X.ndim != 2:
        raise ValueError(
            f"Function {function}: expected a 2D input array, "
            f"but found shape {X.shape}."
        )

    if y.ndim != 1:
        raise ValueError(
            f"Function {function}: expected a 1D output array, "
            f"but found shape {y.shape}."
        )

    if X.shape[1] != expected_dim:
        raise ValueError(
            f"Function {function}: expected {expected_dim} inputs, "
            f"but found {X.shape[1]}."
        )

    if X.shape[0] != y.shape[0]:
        raise ValueError(
            f"Function {function}: input/output row mismatch — "
            f"{X.shape[0]} inputs vs {y.shape[0]} outputs."
        )

    if X.dtype != np.float64:
        raise TypeError(
            f"Function {function}: expected input dtype float64, "
            f"but found {X.dtype}."
        )

    if y.dtype != np.float64:
        raise TypeError(
            f"Function {function}: expected output dtype float64, "
            f"but found {y.dtype}."
        )


    # --------------------------------------------------------
    # 4. BUILD FUNCTION DATAFRAME
    # --------------------------------------------------------

    feature_cols = [
        f"x{i}"
        for i in range(1, expected_dim + 1)
    ]

    function_df = pd.DataFrame(
        X,
        columns=feature_cols
    )

    function_df.insert(
        0,
        "function",
        function
    )

    function_df.insert(
        0,
        "round",
        0
    )

    function_df["y"] = y

    initial_frames.append(
        function_df
    )


# ------------------------------------------------------------
# 5. COMBINE ALL FUNCTIONS
# ------------------------------------------------------------

df_initial = pd.concat(
    initial_frames,
    ignore_index=True,
    sort=False
)


# ------------------------------------------------------------
# 6. FINAL SANITY CHECK
# ------------------------------------------------------------

print(
    f"Loaded {len(df_initial):,} initial observations."
)

print(
    "Functions:",
    sorted(
        df_initial["function"].unique()
    )
)

print("\nObservations by function:")

display(
    df_initial
    .groupby("function")
    .size()
    .rename("observations")
    .to_frame()
)

print("\nInitial dataframe:")

display(
    df_initial.head(15)
)

Loaded 175 initial observations.
Functions: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]

Observations by function:


,observations
function,
1,10
2,10
3,15
4,30
5,20
6,20
7,30
8,40



Initial dataframe:


,round,function,x1,x2,y,x3,x4,x5,x6,x7,x8
0,0,1,0.319404,0.762959,1.322677e-79,NaN,NaN,NaN,NaN,NaN,NaN
1,0,1,0.574329,0.879898,1.033078e-46,NaN,NaN,NaN,NaN,NaN,NaN
2,0,1,0.731024,0.733000,7.710875e-16,NaN,NaN,NaN,NaN,NaN,NaN
3,0,1,0.840353,0.264732,3.341771e-124,NaN,NaN,NaN,NaN,NaN,NaN
4,0,1,0.650114,0.681526,-3.606063e-03,NaN,NaN,NaN,NaN,NaN,NaN
5,0,1,0.410437,0.147554,-2.159249e-54,NaN,NaN,NaN,NaN,NaN,NaN
6,0,1,0.312691,0.078723,-2.089093e-91,NaN,NaN,NaN,NaN,NaN,NaN
7,0,1,0.683418,0.861057,2.535001e-40,NaN,NaN,NaN,NaN,NaN,NaN
8,0,1,0.082507,0.403488,3.606771e-81,NaN,NaN,NaN,NaN,NaN,NaN
9,0,1,0.883890,0.582254,6.229856e-48,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# =============================================================================
# PARSE POST-INITIAL BBO ROIUNDS AND APPEND DATA
# =============================================================================
# Reads all subsequent weekly submissions from
# data/new_data/inputs.txt and outputs.txt, where each round contains
# 8 array(...) input objects (Function 8 may wrap across lines) and
# each output row has 8 values, parsed directly to float64. Assigns
# round numbers sequentially and requires df_initial/FUNCTION_DIMS.

from pathlib import Path
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

NEW_DATA_DIR = Path("../data/new_data")

INPUTS_PATH = NEW_DATA_DIR / "inputs.txt"
OUTPUTS_PATH = NEW_DATA_DIR / "outputs.txt"


# ------------------------------------------------------------
# 2. VALIDATE SOURCE FILES
# ------------------------------------------------------------

if not INPUTS_PATH.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUTS_PATH.resolve()}"
    )

if not OUTPUTS_PATH.exists():
    raise FileNotFoundError(
        f"Output file not found: {OUTPUTS_PATH.resolve()}"
    )


# ------------------------------------------------------------
# 3. READ SOURCE TEXT
# ------------------------------------------------------------

inputs_text = INPUTS_PATH.read_text()
outputs_text = OUTPUTS_PATH.read_text()


# ------------------------------------------------------------
# 4. PARSE INPUT ROUNDS
# ------------------------------------------------------------
# Each logical row is an outer [...] list containing 8
# array([...]) objects.
#
# Function 8 can wrap onto another physical line, so physical
# lines cannot be treated directly as complete rounds.
#
# We accumulate text until square brackets balance.

input_rounds = []

current = []
bracket_depth = 0

for physical_line in inputs_text.splitlines():

    if not physical_line.strip():
        continue

    current.append(physical_line.strip())

    bracket_depth += (
        physical_line.count("[")
        - physical_line.count("]")
    )

    # A complete outer list has now been accumulated.
    if bracket_depth == 0 and current:

        logical_row = " ".join(current)

        # Extract the contents of each array([...]).
        array_strings = re.findall(
            r"array\(\s*\[(.*?)\]\s*\)",
            logical_row,
            flags=re.DOTALL
        )

        parsed_functions = []

        for array_string in array_strings:

            # Parse directly to float64.
            #
            # np.fromstring handles:
            # - ordinary decimals
            # - scientific notation
            # - whitespace
            # - line wrapping
            #
            # No rounding is applied.
            values = np.fromstring(
                array_string,
                dtype=np.float64,
                sep=","
            )

            parsed_functions.append(values)

        input_rounds.append(parsed_functions)

        current = []


# Catch malformed/unbalanced input.
if current or bracket_depth != 0:
    raise ValueError(
        "inputs.txt ended with an incomplete or "
        "unbalanced logical row."
    )


# ------------------------------------------------------------
# 5. PARSE OUTPUT ROUNDS
# ------------------------------------------------------------
# Each physical row contains the 8 function outputs for one round.
#
# np.fromstring converts the textual values directly to float64,
# including values written in scientific notation.

output_rounds = []

for physical_line in outputs_text.splitlines():

    line = physical_line.strip()

    if not line:
        continue

    if not (
        line.startswith("[")
        and line.endswith("]")
    ):
        raise ValueError(
            f"Malformed output row: {line}"
        )

    values = np.fromstring(
        line[1:-1],
        dtype=np.float64,
        sep=","
    )

    output_rounds.append(values)


# ------------------------------------------------------------
# 6. VALIDATE NUMBER OF ROUNDS
# ------------------------------------------------------------

if len(input_rounds) != len(output_rounds):
    raise ValueError(
        f"Input rounds ({len(input_rounds)}) do not match "
        f"output rounds ({len(output_rounds)})."
    )

if len(input_rounds) == 0:
    raise ValueError(
        "No post-initial rounds were found."
    )


# ------------------------------------------------------------
# 7. VALIDATE EACH ROUND
# ------------------------------------------------------------

for round_idx, (round_inputs, round_outputs) in enumerate(
    zip(input_rounds, output_rounds),
    start=1
):

    # Exactly one input array per function.
    if len(round_inputs) != 8:
        raise ValueError(
            f"Round {round_idx}: expected 8 function input arrays, "
            f"found {len(round_inputs)}."
        )

    # Exactly one output per function.
    if len(round_outputs) != 8:
        raise ValueError(
            f"Round {round_idx}: expected 8 outputs, "
            f"found {len(round_outputs)}."
        )

    # Validate dimensionality of each function.
    for function, values in enumerate(
        round_inputs,
        start=1
    ):

        expected_dim = FUNCTION_DIMS[function]

        if values.size != expected_dim:
            raise ValueError(
                f"Round {round_idx}, Function {function}: "
                f"expected {expected_dim} inputs, "
                f"found {values.size}."
            )


# ------------------------------------------------------------
# 8. BUILD POST-INITIAL DATAFRAME
# ------------------------------------------------------------

new_records = []

for round_number, (round_inputs, round_outputs) in enumerate(
    zip(input_rounds, output_rounds),
    start=1
):

    for function in range(1, 9):

        values = round_inputs[
            function - 1
        ]

        record = {
            "round": round_number,
            "function": function,
        }

        # Master schema always contains x1 ... x8.
        # Dimensions not used by lower-dimensional functions
        # remain NaN.
        for i in range(8):

            if i < values.size:
                record[f"x{i + 1}"] = values[i]
            else:
                record[f"x{i + 1}"] = np.nan

        record["y"] = round_outputs[
            function - 1
        ]

        new_records.append(record)


df_new = pd.DataFrame(
    new_records
)


# ------------------------------------------------------------
# 9. ENFORCE MASTER COLUMN ORDER
# ------------------------------------------------------------

MASTER_COLUMNS = [
    "round",
    "function",
    "x1",
    "x2",
    "x3",
    "x4",
    "x5",
    "x6",
    "x7",
    "x8",
    "y",
]

df_new = df_new[
    MASTER_COLUMNS
]

df_initial = df_initial[
    MASTER_COLUMNS
]


# ------------------------------------------------------------
# 10. APPEND TO IMMUTABLE INITIAL DATA
# ------------------------------------------------------------

df_all = pd.concat(
    [
        df_initial,
        df_new,
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 11. FINAL VALIDATION
# ------------------------------------------------------------

expected_new_rows = (
    len(input_rounds) * 8
)

if len(df_new) != expected_new_rows:
    raise RuntimeError(
        f"Expected {expected_new_rows} new rows, "
        f"but created {len(df_new)}."
    )

if len(df_all) != (
    len(df_initial) + len(df_new)
):
    raise RuntimeError(
        "Unexpected row count after append."
    )


# ------------------------------------------------------------
# 12. SUMMARY
# ------------------------------------------------------------

print(
    f"Initial observations: "
    f"{len(df_initial):,}"
)

print(
    f"Post-initial rounds:   "
    f"{len(input_rounds)}"
)

print(
    f"New observations:      "
    f"{len(df_new):,}"
)

print(
    f"Total observations:    "
    f"{len(df_all):,}"
)

print(
    f"Rounds loaded:         "
    f"R1-R{len(input_rounds)}"
)

print("\nObservations by round:")

display(
    df_all
    .groupby("round")
    .size()
    .rename("observations")
    .to_frame()
)

print("\nPost-initial data:")

display(
    df_new
)


Initial observations: 175
Post-initial rounds:   13
New observations:      104
Total observations:    279
Rounds loaded:         R1-R13

Observations by round:


,observations
round,
0,175
1,8
2,8
3,8
4,8
5,8
6,8
7,8
8,8



Post-initial data:


,round,function,x1,x2,x3,x4,x5,x6,x7,x8,y
0,1,1,0.673910,0.620086,NaN,NaN,NaN,NaN,NaN,NaN,-0.015093
1,1,2,0.808081,0.969697,NaN,NaN,NaN,NaN,NaN,NaN,-0.005570
2,1,3,0.413793,0.413793,0.517241,NaN,NaN,NaN,NaN,NaN,-0.012543
3,1,4,0.417570,0.450140,0.372094,0.445315,NaN,NaN,NaN,NaN,-0.137223
4,1,5,0.370205,0.879380,0.719896,1.000000,NaN,NaN,NaN,NaN,1410.187737
...,...,...,...,...,...,...,...,...,...,...,...
99,13,4,0.422624,0.350863,0.359345,0.422000,NaN,NaN,NaN,NaN,0.659991
100,13,5,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,8662.482500
101,13,6,0.435845,0.385066,0.675783,0.748318,0.047503,NaN,NaN,NaN,-0.061120
102,13,7,0.230117,0.105291,0.483789,0.263529,0.313756,0.650578,NaN,NaN,3.283282


In [9]:
# =============================================================================
# SAVE HARMONIZED BBO DATASET TO DISK
# =============================================================================
# Saves the complete dataset (initial data + all subsequent rounds)
# using sufficient precision to round-trip float64 values exactly.

from pathlib import Path

OUTPUT_PATH = Path("../data/data.csv")

# Ensure destination directory exists
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Write float64 values with 17 significant digits.
# This preserves sufficient precision for exact float64 round-tripping.
df_all.to_csv(
    OUTPUT_PATH,
    index=False,
    float_format="%.17g"
)

print(f"Saved {len(df_all):,} observations.")
print(f"Output: {OUTPUT_PATH}")
print(f"File size: {OUTPUT_PATH.stat().st_size:,} bytes")

Saved 279 observations.
Output: ../data/data.csv
File size: 32,567 bytes
